In [4]:
import pandas as pd
import numpy as np

# Dateien finden

In [9]:
import pandas as pd
from pathlib import Path

dateien = sorted(Path("../Data").glob("*.csv"))
print(dateien)

[PosixPath('../Data/Zomato Restaurant Dataset.csv'), PosixPath('../Data/global_ai_jobs_2026.csv'), PosixPath('../Data/viral_videos.csv'), PosixPath('../Data/winequalityN.csv')]


# jede Datei als eigenes DataFrame (für verschiedene Themen)

In [11]:
daten = {}
for pfad in dateien:
    daten[pfad.stem] = pd.read_csv("../Data/" + pfad.name, sep=";")

In [13]:
daten["winequalityN"].head()

,"type,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality"
0,"white,7,0.27,0.36,20.7,0.045,45,170,1.001,3,0...."
1,"white,6.3,0.3,0.34,1.6,0.049,14,132,0.994,3.3,..."
2,"white,8.1,0.28,0.4,6.9,0.05,30,97,0.9951,3.26,..."
3,"white,7.2,0.23,0.32,8.5,0.058,47,186,0.9956,3...."
4,"white,7.2,0.23,0.32,8.5,0.058,47,186,0.9956,3...."


# Imports und Verbindung

In [14]:
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, inspect, text

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}"
    f"@localhost:5433/{os.getenv('POSTGRES_DB')}"
)

# Verbindung testen
with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar())

PostgreSQL 17.11 (Debian 17.11-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


#  Hilfsfunktionen

In [15]:
def spalten_bereinigen(df):
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(" ", "_")
                  .str.replace(r"[^\w]", "", regex=True))
    return df

def tabellenname(pfad):
    return re.sub(r"[^\w]", "_", pfad.stem.lower())

# Alle Dateien laden

In [19]:
DATA_DIR = Path("../Data")
dateien = sorted(DATA_DIR.glob("*.csv"))
print(f"{len(dateien)} Dateien gefunden\n")

for pfad in dateien:
    try:
        df = pd.read_csv(pfad)          # pfad = eine einzelne Datei
        df = spalten_bereinigen(df)
        df = df.drop_duplicates()

        tabelle = tabellenname(pfad)
        df.to_sql(tabelle, engine, if_exists="replace",
                  index=False, chunksize=10000)
        print(f"OK     {pfad.name} -> {tabelle} ({len(df)} Zeilen)")
    except Exception as e:
        print(f"FEHLER {pfad.name}: {e}")

4 Dateien gefunden

OK     Zomato Restaurant Dataset.csv -> zomato_restaurant_dataset (9551 Zeilen)
OK     global_ai_jobs_2026.csv -> global_ai_jobs_2026 (5000 Zeilen)
OK     viral_videos.csv -> viral_videos (12000 Zeilen)
OK     winequalityN.csv -> winequalityn (5329 Zeilen)
